# 05 – Validation of the keyword tags
Run from inside `validation/`. Recomputes every validation number in the report:
1. Precision of the keyword tags, from the human hand-check of 100 reviews (coder 1).
2. Agreement between coder 1 and a second coder (human, when `hand_check_100_coder2.xlsx` is filled in) and an AI coder (Cohen's kappa).
3. Precision and recall of the keyword tags against AI labels on 200 further reviews.
4. Human spot check of 20 AI labels.

**Coding rules:** 1 = the review complains about the category (a mild or secondary complaint counts), 0 = praise, neutral mention, or the word used in another sense. Price/value = complaint that the machine itself is overpriced or not worth the money (repair and warranty costs count as Repair).

In [1]:
import os
import numpy as np
import pandas as pd

def kappa(a, b):
    """Cohen's kappa for two 0/1 label vectors."""
    a, b = np.asarray(a), np.asarray(b)
    po = (a == b).mean()                                   # observed agreement
    pa, pb = a.mean(), b.mean()
    pe = pa * pb + (1 - pa) * (1 - pb)                     # agreement expected by chance
    return np.nan if pe == 1 else (po - pe) / (1 - pe)

def load_check(path):
    """Long table: one row per (review, category) that the keywords tagged, with the coder's 0/1."""
    x = pd.read_excel(path, sheet_name="Check")
    cats = x.columns[6:15]
    long = x.melt(id_vars=["#", "Review ID", "Brand", "Stars"], value_vars=list(cats),
                  var_name="category", value_name="label").dropna(subset=["label"])
    long = long[long.label.astype(str).isin(["0", "1", "0.0", "1.0"])]
    long["label"] = long.label.astype(float).astype(int)
    return long

## 1. Precision of the keyword tags (human coder 1, 100 reviews)

In [2]:
c1 = load_check("hand_check_100_coder1.xlsx")
prec = c1.groupby("category").label.agg(tagged="size", correct="sum")
prec["precision"] = (prec.correct / prec.tagged).round(2)
prec.loc["All"] = [len(c1), c1.label.sum(), round(c1.label.mean(), 2)]
print(prec.sort_values("precision", ascending=False))
print("\nPrecision by star rating:")
print(c1.groupby(np.where(c1.Stars <= 2, "1-2 stars", "4-5 stars")).label.agg(["mean", "size"]).round(2))

                          tagged  correct  precision
category                                            
Delivery / installation      1.0      1.0       1.00
Broke / stopped working     28.0     25.0       0.89
Customer support            11.0      9.0       0.82
Repair / technician         37.0     28.0       0.76
All                        172.0    129.0       0.75
Noise / vibration           22.0     16.0       0.73
Price / value               27.0     19.0       0.70
Cleaning / smell / mould    19.0     13.0       0.68
Design / usability          22.0     15.0       0.68
Return / refund              5.0      3.0       0.60

Precision by star rating:
           mean  size
1-2 stars  0.96   129
4-5 stars  0.12    43


## 2. Inter-coder agreement on the same tags

In [3]:
ai = load_check("hand_check_100_AI_coder.xlsx")
pairs = {"Coder 1 vs AI coder": ai}
if os.path.exists("hand_check_100_coder2.xlsx"):           # second human coder, once filled in
    pairs["Coder 1 vs coder 2"] = load_check("hand_check_100_coder2.xlsx")

for name, other in pairs.items():
    m = c1.merge(other, on=["#", "category"], suffixes=("_1", "_2"))
    rows = [(cat, len(g), (g.label_1 == g.label_2).mean(), kappa(g.label_1, g.label_2))
            for cat, g in m.groupby("category")]
    no_price = m[m.category != "Price / value"]
    rows += [("All", len(m), (m.label_1 == m.label_2).mean(), kappa(m.label_1, m.label_2)),
             ("All except price/value", len(no_price), (no_price.label_1 == no_price.label_2).mean(),
              kappa(no_price.label_1, no_price.label_2))]
    print(f"\n{name}")
    print(pd.DataFrame(rows, columns=["category", "n", "agreement", "kappa"]).round(2).to_string(index=False))


Coder 1 vs AI coder
                category   n  agreement  kappa
 Broke / stopped working  28       0.86   0.42
Cleaning / smell / mould  19       0.95   0.88
        Customer support  11       0.91   0.62
 Delivery / installation   1       0.00   0.00
      Design / usability  22       0.82   0.61
       Noise / vibration  22       0.82   0.49
           Price / value  27       0.56   0.26
     Repair / technician  37       0.92   0.79
         Return / refund   5       0.80   0.55
                     All 172       0.82   0.57
  All except price/value 145       0.87   0.65

Coder 1 vs coder 2
                category   n  agreement  kappa
 Broke / stopped working  28       0.96   0.78
Cleaning / smell / mould  19       0.84   0.62
        Customer support  11       0.82   0.00
 Delivery / installation   1       1.00    NaN
      Design / usability  21       0.90   0.77
       Noise / vibration  22       0.95   0.88
           Price / value  26       0.85   0.58
     Repair / techn

## 3. Keyword tags vs AI labels on 200 further reviews
The AI coder labelled every category for 200 randomly drawn reviews (not in the hand-check sample), so recall can be measured too.
- **Precision:** of the reviews the keywords tag, the share the AI coder also labels as a complaint.
- **Recall:** of the complaints the AI coder finds, the share the keywords catch.

In [4]:
lab = pd.read_csv("ai_labels_200.csv")
kw = pd.read_csv("../data_clean/reviews_laundry_features.csv").set_index("review_id")
cats = [c[3:] for c in lab.columns if c.startswith("ai_")]
rows = []
for c in cats:
    a = lab["ai_" + c].values
    k = kw.loc[lab.review_id, "a_" + c].values
    tp = ((a == 1) & (k == 1)).sum()
    rows.append((c, a.sum(), k.sum(), tp / max(k.sum(), 1), tp / max(a.sum(), 1), kappa(a, k)))
pr = pd.DataFrame(rows, columns=["category", "AI complaints", "keyword tags", "precision", "recall", "kappa"]).round(2)
print(pr.sort_values("recall").to_string(index=False))

        category  AI complaints  keyword tags  precision  recall  kappa
delivery_install              8             3       0.67    0.25   0.35
         support             39            17       0.94    0.41   0.51
design_usability             49            44       0.45    0.41   0.26
     reliability             87            42       0.90    0.44   0.43
  drying_clothes             25            20       0.70    0.56   0.58
        cleaning             34            32       0.66    0.62   0.56
   return_refund             17            14       0.79    0.65   0.69
           noise             30            32       0.66    0.70   0.62
           value             29            56       0.48    0.93   0.55
  repair_service             55            68       0.75    0.93   0.75


## 4. Human spot check of 20 AI labels

In [5]:
sc = pd.read_excel("ai_spot_check_20.xlsx")
agree = pd.to_numeric(sc.iloc[:, -1], errors="coerce")
print(f"Human agrees with the AI label in {int(agree.sum())} of {int(agree.notna().sum())} cases ({agree.mean():.0%})")
print(sc.assign(agree=agree).groupby("AI says").agree.agg(["sum", "count"]))

Human agrees with the AI label in 17 of 20 cases (85%)
                   sum  count
AI says                      
complains           12     14
does NOT complain    5      6
